# Delta Lake Basics

**Dataset:** `samples.bakehouse.sales_transactions`

**Difficulty:** Hard

**Topics:** Delta Lake write/read, append, update, delete, merge (upsert), time travel, history, schema enforcement, schema evolution

Delta Lake is the default table format on Databricks Runtime 8.0+. It adds ACID transactions, scalable metadata handling, and time travel on top of Apache Parquet files.

All Delta tables in this notebook are written to `/tmp/delta_practice/` - a temporary path available in the Databricks environment.

In [ ]:
from pyspark.sql import functions as F, types as T
from delta.tables import DeltaTable

df_transactions = spark.table("samples.bakehouse.sales_transactions")

# Base path for Delta tables in this notebook
base_path = "/tmp/delta_practice"

## Problem 1 - Write and Read a Delta Table

Filter `df_transactions` to rows where `franchiseID == 1`.
Write the result to `{base_path}/transactions_f1` in Delta format using:
```python
.write.format("delta").mode("overwrite").save(path)
```
Then read it back with:
```python
spark.read.format("delta").load(path)
```
Assign the read-back DataFrame to `result_1`.

**Expected columns:** same as `df_transactions`

In [ ]:
# Problem 1 - write your solution here
# Assign result to: result_1

result_1 = None  # replace this

In [ ]:
# -- Tests for Problem 1 --
assert result_1 is not None, "result_1 is None"
assert hasattr(result_1, 'columns'), "Must be a Spark DataFrame"
cnt_1 = result_1.count()
assert cnt_1 > 0, "result_1 has 0 rows - check your filter and write/read"
cols_lower = [c.lower() for c in result_1.columns]
assert 'transactionid' in cols_lower, "Missing column: transactionID"
assert 'franchiseid' in cols_lower, "Missing column: franchiseID"
only_f1 = result_1.filter(F.col('franchiseID') != 1).count()
assert only_f1 == 0, f"Expected only franchiseID=1 rows, found {only_f1} rows with other franchiseIDs"
print(f"Problem 1 passed - ({cnt_1} rows for franchiseID=1)")
result_1.show(5)

## Problem 2 - Append to a Delta Table

Filter `df_transactions` to rows where `franchiseID == 2`.
Append this result to the **same path** `{base_path}/transactions_f1` using mode `append`:
```python
.write.format("delta").mode("append").save(path)
```
Read the table back from the same path and assign to `result_2`.
The table should now contain rows for both `franchiseID` 1 and 2.

**Expected:** `result_2` has more rows than `result_1`, and contains both franchiseID 1 and 2.

In [ ]:
# Problem 2 - write your solution here
# Assign result to: result_2

result_2 = None  # replace this

In [ ]:
# -- Tests for Problem 2 --
assert result_2 is not None, "result_2 is None"
assert hasattr(result_2, 'columns'), "Must be a Spark DataFrame"
cnt_2 = result_2.count()
assert cnt_2 > cnt_1, f"result_2 ({cnt_2} rows) should have more rows than result_1 ({cnt_1} rows) after append"
franchise_ids = [r[0] for r in result_2.select('franchiseID').distinct().collect()]
assert 1 in franchise_ids, "franchiseID=1 not found in result_2"
assert 2 in franchise_ids, "franchiseID=2 not found - append may not have worked"
print(f"Problem 2 passed - ({cnt_2} total rows, franchiseIDs present: {sorted(franchise_ids)})")
result_2.groupBy('franchiseID').count().show()

## Problem 3 - Update Rows in a Delta Table

Use `DeltaTable.forPath(spark, path)` to get a handle on the Delta table at `{base_path}/transactions_f1`.
Use `.update()` to increase `totalPrice` by 10% for all rows where `franchiseID == 1`:
```python
delta_table.update(
    condition=F.col('franchiseID') == 1,
    set={'totalPrice': F.col('totalPrice') * 1.1}
)
```
Read the table back and assign to `result_3`.

**Expected:** `result_3` has the same row count as `result_2`, but `totalPrice` values for `franchiseID=1` are 10% higher than before.

In [ ]:
# Problem 3 - write your solution here
# Assign result to: result_3

result_3 = None  # replace this

In [ ]:
# -- Tests for Problem 3 --
assert result_3 is not None, "result_3 is None"
assert hasattr(result_3, 'columns'), "Must be a Spark DataFrame"
cnt_3 = result_3.count()
assert cnt_3 == cnt_2, f"Row count should not change after update: expected {cnt_2}, got {cnt_3}"
avg_price_f1_updated = result_3.filter(F.col('franchiseID') == 1).agg(F.avg('totalPrice')).collect()[0][0]
avg_price_f1_original = result_1.agg(F.avg('totalPrice')).collect()[0][0]
assert avg_price_f1_updated > avg_price_f1_original, (
    f"Average totalPrice for franchiseID=1 should be higher after 10% increase. "
    f"Before: {avg_price_f1_original:.2f}, After: {avg_price_f1_updated:.2f}"
)
print(f"Problem 3 passed - avg totalPrice for franchiseID=1: {avg_price_f1_original:.2f} -> {avg_price_f1_updated:.2f}")
result_3.show(5)

## Problem 4 - Delete Rows from a Delta Table

First, find any product name that exists in the table:
```python
product_to_delete = spark.read.format("delta").load(path).select('product').first()[0]
```
Then use `DeltaTable.forPath(spark, path).delete()` to remove all rows with that product:
```python
delta_table.delete(F.col('product') == product_to_delete)
```
Read the table back and assign to `result_4`.

**Expected:** `result_4` has fewer rows than `result_3`, and the deleted product is no longer present.

In [ ]:
# Problem 4 - write your solution here
# Assign result to: result_4
# Also assign the deleted product name to: product_to_delete

product_to_delete = None  # replace this
result_4 = None  # replace this

In [ ]:
# -- Tests for Problem 4 --
assert result_4 is not None, "result_4 is None"
assert product_to_delete is not None, "product_to_delete is None"
assert hasattr(result_4, 'columns'), "Must be a Spark DataFrame"
cnt_4 = result_4.count()
assert cnt_4 < cnt_3, f"result_4 ({cnt_4} rows) should have fewer rows than result_3 ({cnt_3} rows) after delete"
remaining = result_4.filter(F.col('product') == product_to_delete).count()
assert remaining == 0, f"Product '{product_to_delete}' still found in result_4 ({remaining} rows) - delete did not work"
print(f"Problem 4 passed - deleted product '{product_to_delete}', rows: {cnt_3} -> {cnt_4}")
result_4.groupBy('product').count().orderBy('count', ascending=False).show(10)

## Problem 5 - MERGE (Upsert)

Create a small updates DataFrame with one new row:
```python
updates = spark.createDataFrame([
    (9999, 1, "Croissant", 999.99, "2024-01-01", "Credit Card"),
], ["transactionID", "franchiseID", "product", "totalPrice", "dateTime", "paymentMethod"])
```
Use `DeltaTable.forPath(spark, path).merge()` to upsert this row into the Delta table:
- Match on `target.transactionID = source.transactionID`
- When matched: update all columns
- When not matched: insert all columns

Read the table back and assign to `result_5`.

**Expected:** `result_5` contains a row with `transactionID = 9999`.

In [ ]:
# Problem 5 - write your solution here
# Assign result to: result_5

result_5 = None  # replace this

In [ ]:
# -- Tests for Problem 5 --
assert result_5 is not None, "result_5 is None"
assert hasattr(result_5, 'columns'), "Must be a Spark DataFrame"
cnt_5 = result_5.count()
assert cnt_5 > 0, "result_5 has 0 rows"
new_row = result_5.filter(F.col('transactionID') == 9999).count()
assert new_row == 1, f"Expected 1 row with transactionID=9999 after merge, found {new_row}"
new_row_data = result_5.filter(F.col('transactionID') == 9999).select('product', 'totalPrice').first()
assert new_row_data['product'] == 'Croissant', f"Expected product='Croissant', got '{new_row_data['product']}'"
assert abs(float(new_row_data['totalPrice']) - 999.99) < 0.01, f"Expected totalPrice=999.99, got {new_row_data['totalPrice']}"
print(f"Problem 5 passed - merge inserted transactionID=9999, total rows: {cnt_5}")
result_5.filter(F.col('transactionID') == 9999).show()

## Problem 6 - Time Travel (Read an Earlier Version)

Delta Lake automatically versions every write operation. Use time travel to read the **original** version of the table (version 0 - the first overwrite write from Problem 1):
```python
spark.read.format("delta").option("versionAsOf", 0).load(path)
```
Assign to `result_6`.

**Expected:** `result_6` contains only rows for `franchiseID=1` (the original write), with unmodified `totalPrice` values.

In [ ]:
# Problem 6 - write your solution here
# Assign result to: result_6

result_6 = None  # replace this

In [ ]:
# -- Tests for Problem 6 --
assert result_6 is not None, "result_6 is None"
assert hasattr(result_6, 'columns'), "Must be a Spark DataFrame"
cnt_6 = result_6.count()
assert cnt_6 > 0, "result_6 has 0 rows - time travel read may have failed"
other_franchise_rows = result_6.filter(F.col('franchiseID') != 1).count()
assert other_franchise_rows == 0, (
    f"Version 0 should only contain franchiseID=1 rows, but found {other_franchise_rows} rows with other franchiseIDs. "
    "Make sure you are reading versionAsOf=0."
)
avg_v0 = result_6.agg(F.avg('totalPrice')).collect()[0][0]
assert avg_v0 < avg_price_f1_updated, "Version 0 prices should be lower than the updated prices from Problem 3"
print(f"Problem 6 passed - version 0 has {cnt_6} rows (franchiseID=1 only), avg totalPrice={avg_v0:.2f}")
result_6.show(5)

## Problem 7 - DESCRIBE HISTORY

Retrieve the full history of operations on the Delta table using either:
```python
DeltaTable.forPath(spark, path).history()
```
or:
```python
spark.sql(f"DESCRIBE HISTORY delta.`{base_path}/transactions_f1`")
```
Assign the history DataFrame to `result_7`.

**Expected columns:** `version`, `timestamp`, `operation`

**Expected:** at least 3 rows - one for each write, append, update, delete, and merge operation performed in problems 1-5.

In [ ]:
# Problem 7 - write your solution here
# Assign result to: result_7

result_7 = None  # replace this

In [ ]:
# -- Tests for Problem 7 --
assert result_7 is not None, "result_7 is None"
assert hasattr(result_7, 'columns'), "Must be a Spark DataFrame"
cols_lower = [c.lower() for c in result_7.columns]
assert 'version' in cols_lower, "Missing column: version"
assert 'timestamp' in cols_lower, "Missing column: timestamp"
assert 'operation' in cols_lower, "Missing column: operation"
cnt_7 = result_7.count()
assert cnt_7 >= 3, f"Expected at least 3 history entries (write, append, update/delete/merge), got {cnt_7}"
operations = [r[0] for r in result_7.select('operation').collect()]
assert 'WRITE' in operations or 'CREATE TABLE AS SELECT' in operations, (
    f"Expected a WRITE operation in history, found: {operations}"
)
print(f"Problem 7 passed - {cnt_7} history entries found")
result_7.select('version', 'timestamp', 'operation').orderBy('version').show(truncate=False)

## Problem 8 - Schema Enforcement and Evolution

Delta Lake enforces schema by default. This problem has two parts:

**Part A - Schema Enforcement:** Try to append a DataFrame with an extra column `discount` to the existing table. Catch the resulting exception to confirm schema enforcement is working:
```python
try:
    df_with_extra_col.write.format("delta").mode("append").save(path)
    enforcement_caught = False
except Exception:
    enforcement_caught = True
```

**Part B - Schema Evolution:** Use `.option("mergeSchema", "true")` to allow the new column to be added:
```python
df_with_extra_col.write.format("delta").mode("append").option("mergeSchema", "true").save(path)
```

Create a new DataFrame with the same schema as `df_transactions` plus a `discount` column (use `F.lit(0.05)` for the value), then perform the two-part operation above.
Read the final table back and assign to `result_8`.

**Expected:** `enforcement_caught == True`, and `result_8` has a `discount` column (with `null` values for old rows and `0.05` for the newly appended rows).

In [ ]:
# Problem 8 - write your solution here
# Assign result to: result_8
# Also assign: enforcement_caught = True/False

enforcement_caught = None  # replace this
result_8 = None  # replace this

In [ ]:
# -- Tests for Problem 8 --
assert enforcement_caught is not None, "enforcement_caught is None - did you run Part A?"
assert enforcement_caught is True, (
    "enforcement_caught should be True - Delta Lake should have raised an exception when appending "
    "a DataFrame with an extra column without mergeSchema=true"
)
assert result_8 is not None, "result_8 is None"
assert hasattr(result_8, 'columns'), "Must be a Spark DataFrame"
cnt_8 = result_8.count()
assert cnt_8 > 0, "result_8 has 0 rows"
cols_lower = [c.lower() for c in result_8.columns]
assert 'discount' in cols_lower, (
    "'discount' column not found in result_8. Make sure you used mergeSchema=true and read the table back."
)
null_discount_count = result_8.filter(F.col('discount').isNull()).count()
non_null_discount_count = result_8.filter(F.col('discount').isNotNull()).count()
assert null_discount_count > 0, "Expected some rows with null discount (old rows before schema evolution)"
assert non_null_discount_count > 0, "Expected some rows with non-null discount (new rows appended with mergeSchema)"
print(f"Problem 8 passed - schema enforcement worked, mergeSchema added 'discount' column")
print(f"  Rows with null discount (old rows): {null_discount_count}")
print(f"  Rows with non-null discount (new rows): {non_null_discount_count}")
result_8.select('transactionID', 'franchiseID', 'product', 'totalPrice', 'discount').show(10)